In [ ]:
import os
from datasets import load_dataset
from PIL import Image
from matplotlib import pyplot as plt
import pandas as pd
import open_clip
import numpy as np
import base64
import io
from IPython.display import HTML, display

from langchain.schema import Document
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain+teddynote.models import MultiModal
from langchain_openai import ChatOpenAI

In [ ]:
# COCO 데이터셋

dataset = load_dataset(path="detection-datasets/coco", name="default", split="train", streaming=True)

In [ ]:
 이미지 저장 폴더와 이미지 개수 설정
IMAGE_FOLDER = "tmp"
N_IMAGES = 20

# 그래프 플로팅을 위한 설정
plot_cols = 5
plot_rows = N_IMAGES // plot_cols
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(plot_rows * 2, plot_cols * 2))
axes = axes.flatten()

# 이미지를 폴더에 저장하고 그래프에 표시
dataset_iter = iter(dataset)
os.makedirs(IMAGE_FOLDER, exist_ok=True)
for i in range(N_IMAGES):
    # 데이터셋에서 이미지와 레이블 추출
    data = next(dataset_iter)
    image = data["image"]
    label = data["objects"]["category"][0]  # 첫 번째 객체의 카테고리를 레이블로 사용

    # 그래프에 이미지 표시 및 레이블 추가
    axes[i].imshow(image)
    axes[i].set_title(label, fontsize=8)
    axes[i].axis("off")

    # 이미지 파일로 저장
    image.save(f"{IMAGE_FOLDER}/{i}.jpg")

# 그래프 레이아웃 조정 및 표시
plt.tight_layout()
plt.show()

멀티모달 임베딩

In [ ]:
pd.DataFrame(open_clip.list_pretrained(), columns=["model_name", "checkpoint"]).head(10)

In [ ]:
image_embedding_function = OpenCLIPEmbeddings(model_name="ViT-H-14-378-quickgelu", checkpoint="dfn5b")

In [ ]:
# 이미지의 경로를 리스트로 저장
image_uris = sorted([os.path.join("tmp", image_name) for image_name in os.listdir("tmp") if image_name.endswith(".jpg")])

image_uris

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

model = MultiModal(
    model=llm, 
    system_prompt="Your mission is to describe the image in detail", 
    user_prompt="Description should be written in one sentence(less than 60 characters)"
)

이미지에 대한 설명 생성

In [ ]:
model.invoke(image_urls[0])

In [ ]:
descriptions = dict()

for i in iamge_uris:
    descriptions[i] = model.invoke(i, display_image=False)

In [ ]:
descriptions

In [ ]:
# 원본 이미지, 처리된 이미지, 텍스트 설명을 저장할 리스트 초기화
original_images = []
images = []
texts = []

# 그래프 크기 설정 (20x10 인치)
plt.figure(figsize=(20, 10))

# 'tmp' 디렉토리에 저장된 이미지 파일들을 처리
for i, image_uri in enumerate(image_uris):
    # 이미지 파일 열기 및 RGB 모드로 변환
    image = Image.open(image_uri).convert("RGB")

    # 4x5 그리드의 서브플롯 생성
    plt.subplot(4, 5, i + 1)

    # 이미지 표시
    plt.imshow(image)

    # 이미지 파일명과 설명을 제목으로 설정
    plt.title(f"{os.path.basename(image_uri)}\n{descriptions[image_uri]}", fontsize=8)

    # x축과 y축의 눈금 제거
    plt.xticks([])
    plt.yticks([])

    # 원본 이미지, 처리된 이미지, 텍스트 설명을 각 리스트에 추가
    original_images.append(image)
    images.append(image)
    texts.append(descriptions[image_uri])

# 서브플롯 간 간격 조정
plt.tight_layout()

아래는 생성한 이미지 description 과 텍스트 간의 유사도를 계산합니다.

In [ ]:
# 이미지와 텍스트 임베딩
img_features = image_embedding_function.embed_image(image_uris)

# 이미지 URI를 사용하여 이미지 특징 추출
text_features = image_embedding_function.embed_documents(
    ["This is " + desc for desc in texts]  # 텍스트 설명에 "This is" 접두사를 추가하고 텍스트 특징 추출
)

# 행렬 연산을 위해 리스트를 numpy 배열로 변환
img_features_np = np.array(img_features)
text_features_np = np.array(text_features)

# 텍스트와 이미지 특징 간의 코사인 유사도를 계산
similarity = np.matmul(text_features_np, img_features_np.T)

텍스트 대 이미지 description 간 유사도를 구하고 시각화합니다.

In [ ]:
# 유사도 행렬을 시각화하기 위한 플롯 생성
count = len(descriptions)
plt.figure(figsize=(20, 14))

# 유사도 행렬을 히트맵으로 표시
plt.imshow(similarity, vmin=0.1, vmax=0.3, cmap="coolwarm")
plt.colorbar()  # 컬러바 추가

# y축에 텍스트 설명 표시
plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])  # x축 눈금 제거

# 원본 이미지를 x축 아래에 표시
for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

# 유사도 값을 히트맵 위에 텍스트로 표시
for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

# 플롯 테두리 제거
for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

# 플롯 범위 설정
plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])

# 제목 추가
plt.title("Cosine Similarity", size=20)

VectorStore 생성 및 이미지 추가

In [ ]:
image_db = Chroma(  # DB 생성성
    collection_name="multimodal", 
    embedding_function=image+embedding_function
)

In [ ]:
image_db.add_images(uris=image_uris)

In [ ]:
# 이미지 검색된 결과를 이미지로 출력하기

class ImageRetriever:
    def __init__(self, retriever):
        """
        이미지 검색기를 초기화합니다.

        인자:
        retriever: LangChain의 retriever 객체
        """
        self.retriever = retriever

    def invoke(self, query):
        """
        쿼리를 사용하여 이미지를 검색하고 표시합니다.

        인자:
        query (str): 검색 쿼리
        """
        docs = self.retriever.invoke(query)
        if docs and isinstance(docs[0], Document):
            self.plt_img_base64(docs[0].page_content)
        else:
            print("검색된 이미지가 없습니다.")
        return docs

    @staticmethod
    def resize_base64_image(base64_string, size=(224, 224)):
        """
        Base64 문자열로 인코딩된 이미지의 크기를 조정합니다.

        인자:
        base64_string (str): 원본 이미지의 Base64 문자열.
        size (tuple): (너비, 높이)로 표현된 원하는 이미지 크기.

        반환:
        str: 크기가 조정된 이미지의 Base64 문자열.
        """
        img_data = base64.b64decode(base64_string)
        img = Image.open(io.BytesIO(img_data))
        resized_img = img.resize(size, Image.LANCZOS)
        buffered = io.BytesIO()
        resized_img.save(buffered, format=img.format)
        return base64.b64encode(buffered.getvalue()).decode("utf-8")

    @staticmethod
    def plt_img_base64(img_base64):
        """
        Base64로 인코딩된 이미지를 표시합니다.

        인자:
        img_base64 (str): Base64로 인코딩된 이미지 문자열
        """
        image_html = f'<img src="data:image/jpeg;base64,{img_base64}" />'
        display(HTML(image_html))

In [ ]:
image_retriever = ImageRetriever(image_db.as_retriever(search_kwargs={"k": 3}))

In [ ]:
# 이미지 조회
result = image_retriever.invoke("A Dog on the street")

In [ ]:
result = image_retriever.invoke("Motorcycle with a man")